In [78]:
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler

# Load and pre-process data

In [111]:
target_col = "log_average_ms"

feature_cols = ["conv_flops", "matmul_flops",
                  "elementwise_mb", "reduction_mb", "normalization_mb",
                  "movement_mb", "memory_mb", "l1d_cache_kb",
                  "l1i_cache_kb", "l2_cache_kb", "base_clock_mhz",
                  "num_cores", "memory_bandwith_gbs"]

metadata_cols = ["model", "cpu_provider",
                 "machine_type", "platform", "run_id"]

In [112]:
# load test, train, val sets

train_df = pd.read_csv("train_set.csv")
val_df = pd.read_csv("val_set.csv")
test_df = pd.read_csv("test_set.csv")

print("train:", train_df.shape)
print("val:", val_df.shape)
print("test:", test_df.shape)

train: (100252, 154)
val: (21483, 154)
test: (21483, 154)


In [113]:
assert list(train_df.columns) == list(val_df.columns) == list(test_df.columns)

train_df.head()

,model,input_dimensions,input_dtypes,output_dimensions,output_dtypes,conv_flops,matmul_flops,elementwise_mb,reduction_mb,normalization_mb,...,base_clock_mhz,memory_bandwith_gbs,cpu_provider,machine_type,platform,repo_file,average_ms,stddev_ms,min_ms,max_ms
0,hardcorenas_d_Opset17_extended.onnx,x:1x3x224x224,x:float32,668:1x1000,668:float32,472440512,2560000,50.528229,5.204269,0.000000,...,2300.000,717,intel,xeon_plat,bluehive,hardcorenas_d_Opset17.onnx,14.069188,0.082284,13.931521,14.247209
1,vit_base_patch8_224_in21k_Opset17_disable_all....,x:1x3x224x224,x:float32,1089:1x21843,1089:float32,231211008,156097479168,1771.169711,0.000000,792.141724,...,2450.000,205,amd,epyc,gcloud,vit_base_patch8_224_in21k_Opset17.onnx,1481.456045,150.199862,1094.324125,1632.506909
2,tf_efficientnetv2_m_in21ft1k_Opset17_basic.onnx,x:1x3x384x384,x:float32,2466:1x1000,2466:float32,31470991360,2560000,1201.339462,67.736023,0.000000,...,2449.998,205,amd,epyc,gcloud,tf_efficientnetv2_m_in21ft1k_Opset17.onnx,301.127424,12.889699,286.464729,324.336829
3,shufflenet_v2_x1_5_Opset16.onnx,x:1x3x224x224,x:float32,1137:1x1000,1137:float32,266034528,2048000,8.246674,1.630875,0.000000,...,2449.998,205,amd,epyc,gcloud,shufflenet_v2_x1_5_Opset16.onnx,8.161246,0.036559,8.080030,8.197800
4,wide_resnet101_2_Opset16_timm.onnx,x:1x3x224x224,x:float32,971:1x1000,971:float32,45502005248,4096000,252.656250,4.218750,0.000000,...,2300.000,717,intel,xeon_plat,bluehive,wide_resnet101_2_Opset16_timm.onnx,38.147223,0.068437,38.035275,38.292409


In [114]:
# pre-processing function

def preprocess(df: pd.DataFrame, train=False) -> pd.DataFrame:

  #drop rows with missing features
  df = df.dropna(subset=["average_ms"])

  #remove rows with high standard deviation (only for train)
  cv_limit = 0.1 # 10% cv limit
  df["cv"] = df["stddev_ms"] / df["average_ms"]

  if train:
    df = df[df["cv"] <= cv_limit].copy()

  #add log_average_ms column
  df["log_average_ms"] = np.log(df["average_ms"])

  #remove decimals and round down
  for col in ["elementwise_mb", "reduction_mb", "normalization_mb", "movement_mb"]:
    df[col] = df[col].round(0)

  return df

In [115]:
train_df = preprocess(train_df, True)
test_df = preprocess(test_df, False)
val_df = preprocess(val_df, False)

print("train:", train_df.shape)
print("val:", val_df.shape)
print("test:", test_df.shape)

train: (95814, 156)
val: (21483, 156)
test: (21483, 156)


# Model Training

In [101]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [102]:
X_train = train_df[feature_cols]
y_train = train_df[target_col]

X_val = val_df[feature_cols]
y_val = val_df[target_col]

X_test = test_df[feature_cols]
y_test = test_df[target_col]


In [86]:
def rmse_log(y_true, y_pred):
  return np.sqrt(mean_squared_error(y_true, y_pred))

In [87]:
def train_xgb(params, X_train, y_train, X_val, y_val):
  model = XGBRegressor(
      objective="reg:squarederror",
      tree_method="hist",
      eval_metric="rmse",
      random_state=67,
      n_jobs=-1,
      early_stopping_rounds=100,
      **params,
  )

  model.fit(
      X_train,
      y_train,
      eval_set=[(X_val, y_val)],
      verbose=100,
  )

  val_pred = model.predict(X_val)
  score = rmse_log(y_val, val_pred)

  return model, score

In [126]:
from sklearn.model_selection import ParameterSampler

param_dist = {
    "n_estimators": [1800, 2000, 2400, 2800, 3000, 3500, 4000, 5000],
    "learning_rate": [0.025, 0.03, 0.04, 0.05, 0.06, 0.08],
    "max_depth": [5, 6, 7, 8],
    "min_child_weight": [3, 5, 8, 10, 15, 20, 30],
    "subsample": [0.85, 0.9, 0.95, 1.0],
    "colsample_bytree": [0.8, 0.85, 0.9, 0.95, 1.0],
    "reg_lambda": [0.5, 1.0, 2.0, 3.0, 5.0, 10.0, 20.0],
    "reg_alpha": [0.0, 0.05, 0.1, 0.3, 0.5, 1.0, 2.0],
}

In [125]:
manual_params = [
    {
        "n_estimators": 2000,
        "learning_rate": 0.08,
        "max_depth": 7,
        "min_child_weight": 10,
        "subsample": 1.0,
        "colsample_bytree": 0.9,
        "reg_lambda": 1.0,
        "reg_alpha": 1.0,
    },
    {
        "n_estimators": 2400,
        "learning_rate": 0.06,
        "max_depth": 7,
        "min_child_weight": 10,
        "subsample": 1.0,
        "colsample_bytree": 0.9,
        "reg_lambda": 1.0,
        "reg_alpha": 1.0,
    },
    {
        "n_estimators": 1800,
        "learning_rate": 0.10,
        "max_depth": 7,
        "min_child_weight": 10,
        "subsample": 1.0,
        "colsample_bytree": 0.9,
        "reg_lambda": 1.0,
        "reg_alpha": 1.0,
    },
    {
        "n_estimators": 2200,
        "learning_rate": 0.08,
        "max_depth": 8,
        "min_child_weight": 15,
        "subsample": 1.0,
        "colsample_bytree": 0.9,
        "reg_lambda": 1.0,
        "reg_alpha": 1.0,
    },
    {
        "n_estimators": 2000,
        "learning_rate": 0.08,
        "max_depth": 7,
        "min_child_weight": 10,
        "subsample": 0.95,
        "colsample_bytree": 0.9,
        "reg_lambda": 1.0,
        "reg_alpha": 1.0,
    },
]

In [127]:
sampled_params = manual_params + list(ParameterSampler(
    param_dist,
    n_iter=60,
    random_state=67,
))

results = []

best_model = None
best_score = float("inf")
best_params = None

for i, params in enumerate(sampled_params, start=1):
  model, val_rmse = train_xgb(params, X_train, y_train, X_val, y_val)

  row = {
        "run": i,
        "val_rmse_log": val_rmse,
        **params,
    }

  results.append(row)

  if val_rmse < best_score:
    best_score = val_rmse
    best_model = model
    best_params = params

  print(f"{i:02d} | val RMSE log = {val_rmse:.5f} | best = {best_score:.5f}")


tuning_results = pd.DataFrame(results).sort_values("val_rmse_log")
tuning_results.head(10)


[0]	validation_0-rmse:1.38362
[100]	validation_0-rmse:0.37875
[200]	validation_0-rmse:0.32042
[300]	validation_0-rmse:0.30158
[400]	validation_0-rmse:0.29546
[500]	validation_0-rmse:0.29257
[600]	validation_0-rmse:0.29101
[700]	validation_0-rmse:0.29005
[800]	validation_0-rmse:0.28952
[900]	validation_0-rmse:0.28906
[1000]	validation_0-rmse:0.28891
[1100]	validation_0-rmse:0.28880
[1200]	validation_0-rmse:0.28878
[1300]	validation_0-rmse:0.28873
[1400]	validation_0-rmse:0.28885
[1430]	validation_0-rmse:0.28889
01 | val RMSE log = 0.28872 | best = 0.28872
[0]	validation_0-rmse:1.40681
[100]	validation_0-rmse:0.40912
[200]	validation_0-rmse:0.34261
[300]	validation_0-rmse:0.31544
[400]	validation_0-rmse:0.30247
[500]	validation_0-rmse:0.29645
[600]	validation_0-rmse:0.29377
[700]	validation_0-rmse:0.29221
[800]	validation_0-rmse:0.29101
[900]	validation_0-rmse:0.29028
[1000]	validation_0-rmse:0.28970
[1100]	validation_0-rmse:0.28931
[1200]	validation_0-rmse:0.28898
[1300]	validation_0-rm

,run,val_rmse_log,n_estimators,learning_rate,max_depth,min_child_weight,subsample,colsample_bytree,reg_lambda,reg_alpha
1,2,0.288553,2400,0.060,7,10,1.00,0.90,1.0,1.00
11,12,0.288581,4000,0.050,8,8,1.00,0.90,0.5,2.00
0,1,0.288724,2000,0.080,7,10,1.00,0.90,1.0,1.00
2,3,0.288909,1800,0.100,7,10,1.00,0.90,1.0,1.00
43,44,0.288952,4000,0.025,7,30,1.00,0.95,2.0,0.10
3,4,0.288990,2200,0.080,8,15,1.00,0.90,1.0,1.00
42,43,0.289038,1800,0.080,7,20,1.00,0.90,1.0,0.30
52,53,0.289047,3000,0.060,7,30,1.00,0.85,3.0,0.05
47,48,0.289298,1800,0.060,7,5,1.00,1.00,10.0,0.05
55,56,0.289311,2400,0.080,7,10,0.95,0.90,3.0,2.00


# Evaluation

In [128]:
final_model = best_model

test_pred = final_model.predict(X_test)
test_rmse_log = np.sqrt(mean_squared_error(y_test, test_pred))

print("Test RMSE: ", test_rmse_log)

Test RMSE:  0.2825057870915487


In [129]:
def latency_metrics_from_log(y_log_true, y_log_pred):
    true_ms = np.exp(y_log_true)
    pred_ms = np.exp(y_log_pred)

    error_ms = pred_ms - true_ms
    rel_err = np.abs(error_ms) / true_ms
    ratio_err = np.maximum(pred_ms / true_ms, true_ms / pred_ms)

    rmse_log = np.sqrt(mean_squared_error(y_log_true, y_log_pred))
    rmse_ms = np.sqrt(np.mean(error_ms ** 2))
    rmse_percent = np.sqrt(np.mean(((error_ms / true_ms) * 100) ** 2))

    return {
        "rmse_log": rmse_log,
        "rmse_ms": rmse_ms,
        "rmse_percent": rmse_percent,

        "median_relative_error": np.median(rel_err),
        "p90_relative_error": np.percentile(rel_err, 90),
        "p95_relative_error": np.percentile(rel_err, 95),

        "median_percent_error": np.median(rel_err) * 100,
        "p90_percent_error": np.percentile(rel_err, 90) * 100,
        "p95_percent_error": np.percentile(rel_err, 95) * 100,

        "median_ratio_error": np.median(ratio_err),
        "p90_ratio_error": np.percentile(ratio_err, 90),

        "within_10pct": np.mean(rel_err <= 0.10),
        "within_25pct": np.mean(rel_err <= 0.25),
        "within_50pct": np.mean(rel_err <= 0.50),
        "within_2x": np.mean(ratio_err <= 2.0),
    }

In [130]:
val_metrics = latency_metrics_from_log(y_val, final_model.predict(X_val))
test_metrics = latency_metrics_from_log(y_test, final_model.predict(X_test))

pd.DataFrame([val_metrics, test_metrics], index=["val", "test"])

,rmse_log,rmse_ms,rmse_percent,median_relative_error,p90_relative_error,p95_relative_error,median_percent_error,p90_percent_error,p95_percent_error,median_ratio_error,p90_ratio_error,within_10pct,within_25pct,within_50pct,within_2x
val,0.288553,146.437448,39.259949,0.079280,0.413951,0.593755,7.928023,41.395085,59.375514,1.083104,1.524546,0.567472,0.812317,0.924731,0.957734
test,0.282506,132.612615,36.069926,0.078311,0.414712,0.590687,7.831146,41.471205,59.068737,1.081896,1.523052,0.572499,0.813853,0.926174,0.957967


# Analysis

In [106]:
test_results = test_df.copy()

test_results["pred_log_latency"] = test_pred
test_results["true_log_latency"] = y_test

test_results["pred_latency_ms"] = np.exp(test_results["pred_log_latency"])
test_results["true_latency_ms"] = np.exp(test_results["true_log_latency"])

test_results["relative_error"] = (
    np.abs(test_results["pred_latency_ms"] - test_results["true_latency_ms"])
    / test_results["true_latency_ms"]
)

In [109]:
def group_latency_metrics(g):
    y_true = g["true_log_latency"].to_numpy()
    y_pred = g["pred_log_latency"].to_numpy()

    true_ms = np.exp(y_true)
    pred_ms = np.exp(y_pred)

    rel_err = np.abs(pred_ms - true_ms) / true_ms

    return pd.Series({
        "count": len(g),
        "rmse_log": np.sqrt(mean_squared_error(y_true, y_pred)),
        "within_10pct": np.mean(rel_err <= 0.10),
        "within_25pct": np.mean(rel_err <= 0.25),
        "within_50pct": np.mean(rel_err <= 0.50),
    })

In [110]:
test_results.groupby("platform").apply(group_latency_metrics).reset_index()

/tmp/ipykernel_6910/3061731483.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  test_results.groupby("platform").apply(group_latency_metrics).reset_index()


,platform,count,rmse_log,within_10pct,within_25pct,within_50pct
0,bluehive,10744.0,0.325766,0.550168,0.778853,0.897710
1,gcloud,10739.0,0.230996,0.600056,0.852314,0.955955


In [119]:
test_results.groupby(["machine_type", "platform"]).apply(group_latency_metrics).reset_index()

/tmp/ipykernel_6910/2224414100.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  test_results.groupby(["machine_type", "platform"]).apply(group_latency_metrics).reset_index()


,machine_type,platform,count,rmse_log,within_10pct,within_25pct,within_50pct
0,epyc,gcloud,9388.0,0.204349,0.622816,0.873349,0.965914
1,xeon,gcloud,1351.0,0.366020,0.441895,0.706144,0.886751
2,xeon_plat,bluehive,10744.0,0.325766,0.550168,0.778853,0.897710


In [121]:
test_results.groupby(["cpu_provider", "platform"]).apply(group_latency_metrics).reset_index()

/tmp/ipykernel_6910/3658681541.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  test_results.groupby(["cpu_provider", "platform"]).apply(group_latency_metrics).reset_index()


,cpu_provider,platform,count,rmse_log,within_10pct,within_25pct,within_50pct
0,amd,gcloud,9388.0,0.204349,0.622816,0.873349,0.965914
1,intel,bluehive,10744.0,0.325766,0.550168,0.778853,0.897710
2,intel,gcloud,1351.0,0.366020,0.441895,0.706144,0.886751


In [122]:
test_results.groupby(["cpu_provider", "platform", "num_cores"]).apply(
    group_latency_metrics,
    include_groups=False
).reset_index()

,cpu_provider,platform,num_cores,count,rmse_log,within_10pct,within_25pct,within_50pct
0,amd,gcloud,1,2636.0,0.149715,0.753414,0.934750,0.980653
1,amd,gcloud,2,4015.0,0.195955,0.619925,0.884184,0.974595
2,amd,gcloud,4,2737.0,0.255572,0.501279,0.798319,0.938984
3,intel,bluehive,1,1347.0,0.241771,0.631032,0.881960,0.945805
4,intel,bluehive,2,2686.0,0.260675,0.656739,0.837305,0.931497
5,intel,bluehive,4,3999.0,0.317943,0.541385,0.775444,0.899725
6,intel,bluehive,8,2712.0,0.418367,0.417404,0.674779,0.837389
7,intel,gcloud,6,1351.0,0.366020,0.441895,0.706144,0.886751
